# Downloading the Data

In [1]:
import importlib.util
import sys

print(f"Python: {sys.version.split()[0]}")
print(f"Interpreter: {sys.executable}")

missing = []
for pkg in ["datasets", "pandas", "pyarrow"]:
    if importlib.util.find_spec(pkg) is None:
        missing.append(pkg)

if missing:
    print("Missing packages:", ", ".join(missing))
    print(f"Run: {sys.executable} -m pip install {' '.join(missing)}")
else:
    print("Environment check passed.")

Python: 3.13.9
Interpreter: /opt/anaconda3/bin/python
Environment check passed.


In [2]:
from datasets import Dataset, DatasetDict, load_dataset
import gzip
import json
import os
import urllib.request

# Define the local directory to save the datasets for cluster upload
SAVE_DIR = "odqa_data"
os.makedirs(SAVE_DIR, exist_ok=True)

SQUAD_TR_VERSION = "1.0.0"
SQUAD_TR_BASE_URL = "https://github.com/boun-tabi/squad-tr/raw/beta/data"
SQUAD_TR_URLS = {
    "train": [
        f"{SQUAD_TR_BASE_URL}/squad-tr-train-v{SQUAD_TR_VERSION}.json.gz",
        f"{SQUAD_TR_BASE_URL}/squad-tr-train-v{SQUAD_TR_VERSION}-excluded.json.gz",
    ],
    "validation": [
        f"{SQUAD_TR_BASE_URL}/squad-tr-dev-v{SQUAD_TR_VERSION}.json.gz",
        f"{SQUAD_TR_BASE_URL}/squad-tr-dev-v{SQUAD_TR_VERSION}-excluded.json.gz",
    ],
}

def _download_json_gz(url):
    with urllib.request.urlopen(url) as response:
        return json.loads(gzip.decompress(response.read()).decode("utf-8"))

def _flatten_squad_articles(articles):
    rows = []
    for article in articles["data"]:
        title = article.get("title", "")
        for paragraph in article["paragraphs"]:
            context = paragraph["context"]
            for qa in paragraph["qas"]:
                rows.append({
                    "id": qa["id"],
                    "title": title,
                    "context": context,
                    "question": qa["question"],
                    "answers": {"text": [answer["text"] for answer in qa["answers"]]},
                })
    return rows

def load_squad_tr_openqa():
    splits = {}
    for split, urls in SQUAD_TR_URLS.items():
        rows = []
        for url in urls:
            rows.extend(_flatten_squad_articles(_download_json_gz(url)))
        splits[split] = Dataset.from_list(rows)
    return DatasetDict(splits)


In [3]:
print("Downloading SQUAD-TR dataset...")

# Load the raw SQuAD-TR files directly; dataset scripts are not needed.
squad_tr_open_qa = load_squad_tr_openqa()

# Display the dataset structure (train and validation splits)
print("SQUAD-TR Structure:")
print(squad_tr_open_qa)


SQUAD-TR Structure:
DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 130319
    })
    validation: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 11873
    })
})


In [4]:
# Show a sample from the training set to verify the content
print("\nSample from SQUAD-TR Training Set:")
print(squad_tr_open_qa['train'][0])


Sample from SQUAD-TR Training Set:
{'id': '56be85543aeaaa14008c9063', 'title': 'Beyonce', 'context': 'Beyoncé Giselle Knowles-Carter (d. 4 Eylül 1981), ABD\'li şarkıcı, söz yazarı, prodüktör ve aktris. Houston, Teksas\'ta doğup büyüdü, çocukken çeşitli şarkı ve dans yarışmalarında sahne aldı ve 1990\'ların sonlarında R&B kız grubu Destiny\'s Child\'ın solisti olarak ün kazandı. Babası Mathew Knowles tarafından yönetilen grup tüm zamanların en çok satan kız gruplarından biri oldu. Beyoncé\'nin ilk albümü Dangerously in Love\'ın (2003) yayınlanmasını izlemiştir ve beş Grammy Ödülü kazanmış ve Billboard Hot 100 bir numaralı single\'ları “Crazy in Love” ve “Baby Boy"un yer aldığı Beyoncé\'nin ilk albümü Dangerously in Love (2003) yayınlandı.', 'question': 'Beyonce ne zaman popüler olmaya başladı?', 'answers': {'text': ["1990'ların sonlarında"]}}


In [5]:
print("Downloading SQUAD-TR dataset...")

# Load the raw SQuAD-TR files directly; dataset scripts are not needed.
squad_tr_open_qa = load_squad_tr_openqa()

# Display the dataset structure (train and validation splits)
print("SQUAD-TR Structure:")
print(squad_tr_open_qa)

# Save to disk
squad_tr_path = os.path.join(SAVE_DIR, "squad_tr")
squad_tr_open_qa.save_to_disk(squad_tr_path)
print(f"Saved SQUAD-TR to {squad_tr_path}")

SQUAD-TR Structure:
DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 130319
    })
    validation: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 11873
    })
})


Saving the dataset (0/1 shards):   0%|          | 0/130319 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/11873 [00:00<?, ? examples/s]

Saved SQUAD-TR to odqa_data/squad_tr


In [6]:
print("Downloading Turkish Wikipedia dump (20230901.tr)...")

# Load the Parquet shards directly; dataset scripts are not needed.
wiki_tr = load_dataset(
    "parquet",
    data_files="hf://datasets/graelo/wikipedia/data/20230901/tr/*.parquet",
    split="train",
)

# Display the dataset structure (should show ~531k rows)
print("Wikipedia TR Structure:")
print(wiki_tr)

# Save to disk
wiki_tr_path = os.path.join(SAVE_DIR, "wiki_20230901_tr")
wiki_tr.save_to_disk(wiki_tr_path)
print(f"Saved Wikipedia TR to {wiki_tr_path}")

data/20230901/tr/train-0001-of-0004.parq(…):   0%|          | 0.00/139M [00:00<?, ?B/s]

data/20230901/tr/train-0002-of-0004.parq(…):   0%|          | 0.00/139M [00:00<?, ?B/s]

data/20230901/tr/train-0003-of-0004.parq(…):   0%|          | 0.00/138M [00:00<?, ?B/s]

data/20230901/tr/train-0004-of-0004.parq(…):   0%|          | 0.00/139M [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Wikipedia TR Structure:
Dataset({
    features: ['id', 'url', 'title', 'text'],
    num_rows: 530830
})


Saving the dataset (0/2 shards):   0%|          | 0/530830 [00:00<?, ? examples/s]

Saved Wikipedia TR to odqa_data/wiki_20230901_tr


# Preprocessing the Dataset

In [7]:
import string
import os

def chunk_wikipedia_passages(batch):
    titles = batch['title']
    texts = batch['text']
    
    chunked_titles = []
    chunked_texts = []
    
    # Paper specification: 75 words per chunk to prevent BERT truncation
    chunk_size = 75 
    
    for title, text in zip(titles, texts):
        # Skip if there is no text to chunk
        if not text:
            continue
            
        # Enhanced Whitespace Tokenizer (applied only to the text)
        # Pad punctuation with spaces so they are treated as distinct tokens
        processed_text = text
        for punc in string.punctuation:
            processed_text = processed_text.replace(punc, f" {punc} ")
            
        # Split purely by whitespace
        tokens = processed_text.split()
        
        # Split into equal chunks of 75 words
        for i in range(0, len(tokens), chunk_size):
            chunk = tokens[i:i + chunk_size]
            
            # Keep the title separate for the DPR tokenizer
            chunked_titles.append(title)
            
            # Join the tokens back into a string chunk
            chunk_str = " ".join(chunk)
            
            # Clean up the aggressive punctuation spacing for readability 
            chunk_str = chunk_str.replace(" .", ".").replace(" ,", ",").replace(" ' ", "'")
            
            chunked_texts.append(chunk_str)
            
    return {"title": chunked_titles, "text": chunked_texts}

print("Chunking Turkish Wikipedia into 75-word segments...")

# Apply the mapping function to the dataset
wiki_tr_chunked = wiki_tr.map(
    chunk_wikipedia_passages, 
    batched=True, 
    remove_columns=wiki_tr.column_names, 
    desc="Chunking Wikipedia passages"
)

print("New Chunked Wikipedia TR Structure:")
print(wiki_tr_chunked)

# Save the preprocessed chunks back to disk
chunked_path = os.path.join(SAVE_DIR, "wiki_20230901_tr_chunked")
wiki_tr_chunked.save_to_disk(chunked_path)
print(f"Saved chunked Wikipedia TR to {chunked_path}")

Chunking Turkish Wikipedia into 75-word segments...


Chunking Wikipedia passages:   0%|          | 0/530830 [00:00<?, ? examples/s]

New Chunked Wikipedia TR Structure:
Dataset({
    features: ['title', 'text'],
    num_rows: 2281587
})


Saving the dataset (0/3 shards):   0%|          | 0/2281587 [00:00<?, ? examples/s]

Saved chunked Wikipedia TR to odqa_data/wiki_20230901_tr_chunked


In [8]:
import pandas as pd
from datasets import Dataset, concatenate_datasets

print("Extracting unique context passages from SQUAD-TR...")

# Convert the train split to a pandas DataFrame for easy deduplication
squad_train_df = squad_tr_open_qa['train'].to_pandas()

# Filter down to just titles and contexts, drop duplicates, and rename 'context' to 'text' 
# so it matches the input format expected by our chunk_wikipedia_passages function
unique_squad_df = squad_train_df[['title', 'context']].drop_duplicates().rename(columns={'context': 'text'})

# Convert back to a Hugging Face Dataset
squad_unique_ds = Dataset.from_pandas(unique_squad_df, preserve_index=False)
print(f"Extracted {len(squad_unique_ds)} unique passages from SQUAD-TR.")

print("Chunking the SQUAD-TR passages into 75-word segments...")
# Apply the exact same chunking function we used on the Wikipedia dump
squad_chunked = squad_unique_ds.map(
    chunk_wikipedia_passages, 
    batched=True, 
    remove_columns=squad_unique_ds.column_names,
    desc="Chunking SQUAD-TR passages"
)

print("Concatenating Wikipedia and SQUAD-TR chunks...")
# Combine both chunked datasets into the final knowledge source
final_knowledge_source = concatenate_datasets([wiki_tr_chunked, squad_chunked])

print("\nFinal Knowledge Source Structure:")
print(final_knowledge_source)

# Save the final, combined, and chunked dataset to disk for the cluster
final_path = os.path.join(SAVE_DIR, "final_knowledge_source_chunked")
final_knowledge_source.save_to_disk(final_path)
print(f"\nSaved final combined knowledge source to {final_path}")
print("Data preparation complete. Ready for offline indexing!")

Extracting unique context passages from SQUAD-TR...
Extracted 19029 unique passages from SQUAD-TR.
Chunking the SQUAD-TR passages into 75-word segments...


Chunking SQUAD-TR passages:   0%|          | 0/19029 [00:00<?, ? examples/s]

Concatenating Wikipedia and SQUAD-TR chunks...

Final Knowledge Source Structure:
Dataset({
    features: ['title', 'text'],
    num_rows: 2320818
})


Saving the dataset (0/3 shards):   0%|          | 0/2320818 [00:00<?, ? examples/s]


Saved final combined knowledge source to odqa_data/final_knowledge_source_chunked
Data preparation complete. Ready for offline indexing!


In [9]:
from datasets import load_from_disk

squad = load_from_disk("odqa_data/squad_tr")
knowledge_source = load_from_disk("odqa_data/final_knowledge_source_chunked")


In [10]:
squad

DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 130319
    })
    validation: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 11873
    })
})

In [11]:
knowledge_source

Dataset({
    features: ['title', 'text'],
    num_rows: 2320818
})